In [37]:
import numpy as np
import pandas as pd
import datetime as dt
from sklearn.cluster import KMeans

In [38]:
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [39]:
nday = 243
Rf = 2.0
n_cluster = 10
n_class = 5
rday = sqrt(nday)

In [40]:
dt_now = dt.datetime.now().date()
dt_now

datetime.date(2024, 6, 30)

In [41]:
# 基金
all_fund = get_all_securities('fund', dt_now)
len(all_fund)

1481

In [42]:
# 一年前上市
dt_1y = dt_now - dt.timedelta(365)
funds = all_fund[all_fund.start_date < dt_1y].index.tolist()
len(funds)

1287

In [43]:
# 流动性
hm = history(nday, '1d', 'money', funds).mean()
funds = hm[hm > 1e6].index.tolist()
len(funds)

804

In [44]:
# 提取历史价格
p = history(nday, '1d', 'close', funds).dropna(axis=1)
r = np.log(p).diff()[1:]

In [45]:
# K-means聚类
cluster = KMeans(n_clusters=n_cluster).fit(r.T)

In [46]:
# 分类
y = cluster.fit_predict(r.T)

In [47]:
# 标签
c = pd.Series(y, r.columns)
c.head()

159001.XSHE    8
159003.XSHE    8
159005.XSHE    8
159501.XSHE    3
159507.XSHE    9
dtype: int32

In [57]:
df = pd.DataFrame(columns=['Cluster', 'Name'])
for k in range(n_cluster):
    choice = []
    for s in c.index:
        if c[s] == k:
            choice.append(s) 
    print(k, len(choice))
    xm = hm[choice].sort_values(ascending=False).head(n_class)
    for s in xm.index:
        df.loc[s] = [k, get_security_info(s).display_name]
len(df)

0 73
1 61
2 186
3 32
4 59
5 38
6 100
7 82
8 92
9 80


50

In [58]:
df.sort_values(by=['Cluster', 'Name'], ascending=True)

,Cluster,Name
512100.XSHG,0,1000ETF
560010.XSHG,0,1000基金
510500.XSHG,0,500ETF
159845.XSHE,0,中证1000ETF
159922.XSHE,0,中证500ETF
513120.XSHG,1,HK创新药
512170.XSHG,1,医疗ETF
512010.XSHG,1,医药ETF
513060.XSHG,1,恒生医疗
159892.XSHE,1,恒生医药ETF
